In [47]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import joblib
from march_madness.config import PROCESSED_DATA_DIR, MODELS_DIR

In [ ]:
df_features = pd.read_parquet(PROCESSED_DATA_DIR / 'df_tn_final.parquet')

In [ ]:
guess_logloss = 0.693

In [ ]:
df_features.dtypes

In [ ]:
max_train_season = df_features['Season'].max()-1
df_train = df_features[df_features['Season']<=max_train_season]
df_test = df_features[df_features['Season']>max_train_season]

In [ ]:
feature_cols = [
 'win_pct_lower',
 'team_fgm3_percent_lower',
 'team_fgm_percent_lower',
 'team_ft_percent_lower',
 'opp_fgm3_percent_lower',
 'opp_fgm_percent_lower',
 'opp_ft_percent_lower',
 'pg_team_Score_lower',
 'pg_team_OR_lower',
 'pg_team_DR_lower',
 'pg_team_Ast_lower',
 'pg_team_TO_lower',
 'pg_team_Stl_lower',
 'pg_team_Blk_lower',
 'pg_team_PF_lower',
 'pg_opp_Score_lower',
 'pg_opp_OR_lower',
 'pg_opp_DR_lower',
 'pg_opp_Ast_lower',
 'pg_opp_TO_lower',
 'pg_opp_Stl_lower',
 'pg_opp_Blk_lower',
 'pg_opp_PF_lower',
 'win_pct_higher',
 'team_fgm3_percent_higher',
 'team_fgm_percent_higher',
 'team_ft_percent_higher',
 'opp_fgm3_percent_higher',
 'opp_fgm_percent_higher',
 'opp_ft_percent_higher',
 'pg_team_Score_higher',
 'pg_team_OR_higher',
 'pg_team_DR_higher',
 'pg_team_Ast_higher',
 'pg_team_TO_higher',
 'pg_team_Stl_higher',
 'pg_team_Blk_higher',
 'pg_team_PF_higher',
 'pg_opp_Score_higher',
 'pg_opp_OR_higher',
 'pg_opp_DR_higher',
 'pg_opp_Ast_higher',
 'pg_opp_TO_higher',
 'pg_opp_Stl_higher',
 'pg_opp_Blk_higher',
 'pg_opp_PF_higher',
 'seed_num_lower',
 'seed_num_higher'
 ]

In [ ]:
target_col = 'lower_id_won'

In [ ]:
m_default_rf = RandomForestClassifier(random_state=42)

In [ ]:
m_default_rf.fit(df_train[feature_cols], df_train[target_col])

In [ ]:
train_preds = m_default_rf.predict_proba(df_train[feature_cols])[:,1]
test_preds = m_default_rf.predict_proba(df_test[feature_cols])[:,1]

In [ ]:
train_log_loss = log_loss(y_true = df_train[target_col], y_pred = train_preds)
test_log_loss = log_loss(y_true = df_test[target_col], y_pred = test_preds)

In [ ]:
print(f"Train Log Loss: {train_log_loss:.4f}")
print(f"Test  Log Loss: {test_log_loss:.4f}")

In [ ]:
num_features = len(feature_cols)

In [ ]:
param_grid = {
    'max_features':      ["sqrt", "log2", None],
    'min_samples_leaf':  [1, 3, 5, 10],
    'max_samples':       [0.5, 0.63, 0.8, 0.95]
}

In [ ]:
rf = RandomForestClassifier(
    n_estimators=1000, 
    criterion='log_loss',
    oob_score=True,
    bootstrap=True,
    random_state=42, 
    n_jobs=-1,
)


In [ ]:
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv = 5,
    n_jobs=-1
)

In [ ]:
# random_search = RandomizedSearchCV(
#     estimator=rf,
#     param_distributions=param_grid,
#     n_iter=50,
#     cv = 5,
#     n_jobs=-1
# )

In [ ]:
grid_search.fit(df_train[feature_cols], df_train[target_col])

In [ ]:
joblib.dump(grid_search, MODELS_DIR/'grid_search.pkl')

['C:\\Users\\leste\\Documents\\GitHub\\march-madness\\models\\random_search.pkl']